# 04 — Run MEFISTO once on the full cohort

**Input:** the same filtered counts and metadata used by TEMPTED  
**Does:** applies pseudocount-free rCLR and fits one five-factor MEFISTO model using all eligible subjects  
**Output:** fitted sample factors, subject mean factor scores, genus loadings, and the saved model

Original zeros stay missing (`NaN`). There is no train/test split or held-out projection here.

In [ ]:
from datetime import datetime
from pathlib import Path

import anndata as ad
import muon as mu
import numpy as np
import pandas as pd

N_FACTORS = 5
N_ITERATIONS = 1000
SEED = 2026

root = Path(".") if Path("data").exists() else Path("..")
input_folder = sorted((root / "data" / "preprocessing").iterdir())[-1]
output = root / "data" / "mefisto" / datetime.now().strftime("%Y%m%d_%H%M%S")
output.mkdir(parents=True)

counts = pd.read_csv(input_folder / "counts_filtered.csv", dtype={"sample_id": str}).set_index("sample_id")
metadata = pd.read_csv(
    input_folder / "metadata.csv",
    dtype={"sample_id": str, "subject_id": str, "country": str},
)
counts = counts.loc[metadata["sample_id"]]

print("Input:", input_folder)
print("Output:", output)

## rCLR

Only positive counts are logged and centered within each sample. Zeros remain missing.

In [ ]:
values = counts.to_numpy(float)
rclr = np.full(values.shape, np.nan)

for i, row in enumerate(values):
    positive = row > 0
    logged = np.log(row[positive])
    rclr[i, positive] = logged - logged.mean()

rclr = pd.DataFrame(rclr, index=counts.index, columns=counts.columns)

## Fit one full-cohort MEFISTO model

In [ ]:
obs = metadata.set_index("sample_id")[["subject_id", "age", "country"]].copy()
obs["model_group"] = pd.factorize(obs["subject_id"])[0].astype(str)

adata = ad.AnnData(
    rclr.loc[obs.index].to_numpy(np.float32),
    obs=obs,
    var=pd.DataFrame(index=rclr.columns.astype(str)),
)

mu.tl.mofa(
    adata,
    groups_label="model_group",
    likelihoods="gaussian",
    center_groups=False,
    n_factors=N_FACTORS,
    n_iterations=N_ITERATIONS,
    convergence_mode="medium",
    smooth_covariate="age",
    smooth_kwargs={"scale_cov": True, "sparseGP": False, "model_groups": False},
    seed=SEED,
    outfile=str(output / "model.hdf5"),
)

In [ ]:
factors = np.asarray(adata.obsm["X_mofa"], float)
loadings = np.asarray(adata.varm["LFs"], float)
names = [f"factor_{i+1}" for i in range(factors.shape[1])]

sample_factors = metadata[["sample_id", "subject_id", "age", "country"]].copy()
sample_factors[names] = factors

subject_scores = (
    sample_factors.groupby("subject_id", as_index=False)
    .agg({**{name: "mean" for name in names}, "country": "first"})
)

feature_loadings = pd.DataFrame(loadings, columns=names)
feature_loadings.insert(0, "feature_id", counts.columns)

if len(names) != N_FACTORS:
    raise ValueError(f"Expected {N_FACTORS} factors; found {len(names)}.")

sample_factors.to_csv(output / "sample_factors.csv", index=False)
subject_scores.to_csv(output / "subject_scores.csv", index=False)
feature_loadings.to_csv(output / "feature_loadings.csv", index=False)
pd.DataFrame({
    "n_factors": [N_FACTORS],
    "n_iterations_requested": [N_ITERATIONS],
    "seed": [SEED],
}).to_csv(output / "settings.csv", index=False)

print("Saved:", output)
display(subject_scores.head())